In [1]:
import tqdm
import warnings
import quantstats as qs
from utils.evaluate import *
from utils.data_bridge import *
from method import *
warnings.filterwarnings('ignore')
qs.extend_pandas()

In [2]:
# 定義開盤漲跌幅和日內漲跌幅
handler = Handler('data')
handler.start_date = '2019-01-01'

adj_close = handler['Close'] * handler['Adjust_Factor']
adj_open = handler['Open'] * handler['Adjust_Factor']

# 隔夜
exp_overnight_ret = adj_open.shift(-2) / adj_close.shift(-1) - 1
# 日內
exp_intrday_ret = adj_close.shift(-1) / adj_open.shift(-1) - 1
# 波段
exp_day_ret = adj_open.shift(-2) / adj_open.shift(-1) - 1

In [ ]:
# 濾網
Close = handler['Close']
Values = handler['Value_Dollars'].shift(1)

# 濾網
上市超過5天 = handler['Close'].notna().astype(int).rolling(5).sum()==5
成交額濾網 = handler['Value_Dollars'] >= 20_000_000
開盤非漲停 = ~((handler['Limit_Up_or_Down_in_Opening_Fg'] == 'Y') & (adj_open > adj_close.shift(1)))
買賣當沖 = handler['Suspension_of_buy_After_Day_Trading_Fg'] != 'Y'
非漲停 = handler['Limit_Up_or_Down'] != '+'

濾網 = 上市超過5天 & 成交額濾網 & 開盤非漲停.shift(-1)
隔夜濾網 = 成交額濾網 & 非漲停.shift(-1)
日內濾網 = 成交額濾網 & 買賣當沖.shift(-1)

In [4]:
# 單因子分析
expr = "cs_rank(Turnover)"
factor = eval(expr, funcs_methods, handler)

# 設定濾網
factor = factor.where(日內濾網, np.nan)
exp_ret = exp_intrday_ret.where(日內濾網, np.nan)
ret = factors_analyze(factor, exp_ret, one_side=False, rank_range_n=10, quantile_metric='mean')

1. 分組統計指標表（bps） — Factor (方向: -)...


,計數,比例(%),平均(bps),中位數(bps),標準差(bps)
X 範: Factor,,,,,
"(-1803, -1601]",107545,9.93,-45.2610,-71.2251,385.9756
"(-1601, -1531]",108392,10.01,-40.5057,-60.8396,321.4884
"(-1531, -1459]",108201,9.99,-36.0282,-52.2193,287.3084
"(-1459, -1381]",108404,10.01,-32.9850,-46.7290,260.3048
"(-1381, -1296]",108539,10.02,-28.2800,-39.8406,237.8948
"(-1296, -1198]",108057,9.98,-25.8057,-35.0877,218.9314
"(-1198, -1081]",108231,9.99,-20.8623,-27.9330,198.2422
"(-1081, -933.000]",108373,10.01,-15.0044,-19.1571,178.0938
"(-933.000, -722.000]",108211,9.99,-9.6645,-11.4679,154.0653



2. 繪製分組 mean 比較圖...


3. 繪製累積收益曲線圖...


統計指標:


,CAGR(%),Sharpe,MDD(%),單利MDD(%),IC,ICIR,樣本勝率(%),周勝率(%),月勝率(%),年勝率(%),盈虧比,總賺賠比,預期報酬(bps),樣本數
Factor,47.22,4.88,-3.63,-1.85,0.1098,0.5607,61.25,76.42,97.65,100.0,1.44,2.27,15.36,1716
